In [6]:
from bravado.client import SwaggerClient
import pandas as pd
import requests
from tqdm.auto import tqdm

In [ ]:
KLIFS_CLIENT = SwaggerClient.from_url(
    "https://klifs.net/swagger/swagger.json", 
    config={"validate_responses": False, "use_models": False}
)

In [3]:
kinases_raw = KLIFS_CLIENT.Information.get_kinase_names().response().result
kinases_df = pd.DataFrame(kinases_raw)
human_kinases = kinases_df[kinases_df["species"] == "Human"].reset_index(drop=True)
print(f"Человеческих киназ: {len(human_kinases)}")

Человеческих киназ: 555


In [5]:
uniprot_data = []

for _, row in tqdm(human_kinases.iterrows(), total=len(human_kinases)):
    try:
        info = KLIFS_CLIENT.Information.get_kinase_information(kinase_ID=[row["kinase_ID"]]).response().result[0]
        uniprot = info.get("uniprot")
        if uniprot and uniprot != "None":
            uniprot_data.append({
                "kinase_name": row["name"],
                "uniprot_id": uniprot
            })
    except Exception:
        continue

uniprot_df = pd.DataFrame(uniprot_data)
print(f"Успешно получено UniProt ID для {len(uniprot_df)} киназ")

  0%|          | 0/555 [00:00<?, ?it/s]

Успешно получено UniProt ID для 524 киназ


In [ ]:
def fetch_chembl_activities(uniprot_id, activity_types=("Kd", "Ki")):

    base = "https://www.ebi.ac.uk/chembl/api/data"
    activities = []

    try:
        target_resp = requests.get(
            f"{base}/target.json",
            params={"target_components__accession": uniprot_id, "format": "json"}
        ).json()
        if not target_resp.get("targets"):
            return []
        target_chembl_id = target_resp["targets"][0]["target_chembl_id"]
    except:
        return []

    for act_type in activity_types:
        url = f"{base}/activity.json"
        page = 1
        while url:
            try:
                resp = requests.get(url, params={
                    "target_chembl_id": target_chembl_id,
                    "standard_type": act_type,
                    "standard_units": "nM",
                    "format": "json",
                    "limit": 1000
                } if page == 1 else None).json()

                for act in resp.get("activities", []):
                    smiles = act.get("canonical_smiles")
                    value = act.get("standard_value")
                    pchembl = act.get("pchembl_value")
                    mol_id = act.get("molecule_chembl_id")
                    if smiles and value and mol_id:
                        try:
                            activities.append({
                                "ligand_chembl_id": mol_id,
                                "smiles": smiles,
                                "activity_type": act_type,
                                "activity_value_nM": float(value),
                                "pchembl_value": float(pchembl) if pchembl else None
                            })
                        except (ValueError, TypeError):
                            continue

                # Пагинация
                next_url = resp.get("page_meta", {}).get("next")
                if next_url:
                    url = "https://www.ebi.ac.uk" + next_url
                    page += 1

                else:
                    break
            except Exception:
                break
    return activities

In [ ]:
all_records = []

for _, row in tqdm(uniprot_df.iterrows(), total=len(uniprot_df)):
    kinase_name = row["kinase_name"]
    uniprot_id = row["uniprot_id"]
    acts = fetch_chembl_activities(uniprot_id)
    for a in acts:
        all_records.append({
            "kinase_name": kinase_name,
            "uniprot_id": uniprot_id,
            "ligand_chembl_id": a["ligand_chembl_id"],
            "smiles": a["smiles"],
            "activity_type": a["activity_type"],
            "activity_value_nM": a["activity_value_nM"],
            "pchembl_value": a["pchembl_value"]
        })

In [ ]:
dataset = pd.DataFrame(all_records)
print(f"\nВсего записей собрано: {len(dataset)}")

if not dataset.empty:
    dataset = dataset.drop_duplicates(subset=["kinase_name", "smiles", "activity_type"]).reset_index(drop=True)
    print(f"После удаления дубликатов: {len(dataset)} записей")

    output_file = "human_kinase_chembl_Kd_Ki.csv"
    dataset.to_csv(output_file, index=False)
    print(f"Датасет сохранён: {output_file}")
    
    print("\nТипы активностей:")
    print(dataset["activity_type"].value_counts())
    
display(dataset.head())